In [ ]:
!pip install ultralytics opencv-python onnx onnxruntime pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.5 MB/s eta 0:00:00


In [ ]:
import cv2
import time
import numpy as np
import pandas as pd
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Load YOLOv8 Nano model
model = YOLO("yolov8n.pt")

print("YOLOv8n model loaded successfully!")

YOLOv8n model loaded successfully!


In [ ]:
video_path = "/content/drive/MyDrive/videos/00000002.mp4"

In [ ]:
def run_inference(
    model,
    video_path,
    output_path,
    img_size=640,
    frame_skip=1
):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error opening video")
        return None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_input = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out = cv2.VideoWriter(
        output_path,
        fourcc,
        fps_input,
        (width, height)
    )

    frame_count = 0
    fps_values = []

    start_total = time.time()

    while True:

        success, frame = cap.read()

        if not success:
            break

        frame_count += 1

        # Frame skipping
        if frame_count % frame_skip != 0:
            continue

        start_time = time.time()

        # YOLO inference on CPU
        results = model.predict(
            frame,
            imgsz=img_size,
            device="cpu",
            verbose=False
        )

        annotated_frame = results[0].plot()

        # FPS calculation
        end_time = time.time()

        fps = 1 / (end_time - start_time)
        fps_values.append(fps)

        # Draw FPS text
        cv2.putText(
            annotated_frame,
            f"FPS: {fps:.2f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        out.write(annotated_frame)

    total_time = time.time() - start_total

    cap.release()
    out.release()

    avg_fps = np.mean(fps_values)
    min_fps = np.min(fps_values)
    max_fps = np.max(fps_values)

    print("Processing completed!")
    print("Output saved:", output_path)

    print(f"Average FPS = {avg_fps:.2f}")
    print(f"Min FPS = {min_fps:.2f}")
    print(f"Max FPS = {max_fps:.2f}")

    return {
        "resolution": img_size,
        "frame_skip": frame_skip,
        "avg_fps": round(avg_fps, 2),
        "min_fps": round(min_fps, 2),
        "max_fps": round(max_fps, 2),
        "processing_time_sec": round(total_time, 2)
    }

In [ ]:
baseline_result = run_inference(
    model=model,
    video_path=video_path,
    output_path="baseline_output.mp4",
    img_size=640,
    frame_skip=1
)

baseline_result

Processing completed!
Output saved: baseline_output.mp4
Average FPS = 6.69
Min FPS = 1.69
Max FPS = 8.22


{'resolution': 640,
 'frame_skip': 1,
 'avg_fps': np.float64(6.69),
 'min_fps': np.float64(1.69),
 'max_fps': np.float64(8.22),
 'processing_time_sec': 23.87}

In [ ]:
results_list = []

results_list.append({
    "Technique": "Baseline",
    "Resolution": "640x640",
    "Frame Skip": "No",
    "Average FPS": baseline_result["avg_fps"],
    "Min FPS": baseline_result["min_fps"],
    "Max FPS": baseline_result["max_fps"]
})

In [ ]:
results_df = pd.DataFrame(results_list)

results_df

,Technique,Resolution,Frame Skip,Average FPS,Min FPS,Max FPS
0,Baseline,640x640,No,6.69,1.69,8.22


PART-2

In [ ]:
resolution_sizes = [640, 416, 320]

resolution_results = []

for size in resolution_sizes:

    print(f"\nRunning inference for {size}x{size}...\n")

    result = run_inference(
        model=model,
        video_path=video_path,
        output_path=f"output_{size}.mp4",
        img_size=size,
        frame_skip=1
    )

    resolution_results.append(result)


Running inference for 640x640...

Processing completed!
Output saved: output_640.mp4
Average FPS = 6.48
Min FPS = 1.12
Max FPS = 7.70

Running inference for 416x416...

Processing completed!
Output saved: output_416.mp4
Average FPS = 14.12
Min FPS = 3.52
Max FPS = 17.13

Running inference for 320x320...

Processing completed!
Output saved: output_320.mp4
Average FPS = 21.91
Min FPS = 10.92
Max FPS = 25.85


In [ ]:
for result in resolution_results:

    results_list.append({
        "Technique": f"Resolution {result['resolution']}",
        "Resolution": f"{result['resolution']}x{result['resolution']}",
        "Frame Skip": "No",
        "Average FPS": result["avg_fps"],
        "Min FPS": result["min_fps"],
        "Max FPS": result["max_fps"]
    })

In [ ]:
results_df = pd.DataFrame(results_list)

results_df

,Technique,Resolution,Frame Skip,Average FPS,Min FPS,Max FPS
0,Baseline,640x640,No,6.69,1.69,8.22
1,Resolution 640,640x640,No,6.48,1.12,7.70
2,Resolution 416,416x416,No,14.12,3.52,17.13
3,Resolution 320,320x320,No,21.91,10.92,25.85


In [ ]:
print("Running frame skipping experiment...")

skip_1_result = run_inference(
    model=model,
    video_path=video_path,
    output_path="frame_skip_1.mp4",
    img_size=416,
    frame_skip=1
)

skip_2_result = run_inference(
    model=model,
    video_path=video_path,
    output_path="frame_skip_2.mp4",
    img_size=416,
    frame_skip=2
)

Running frame skipping experiment...
Processing completed!
Output saved: frame_skip_1.mp4
Average FPS = 15.30
Min FPS = 1.78
Max FPS = 17.26
Processing completed!
Output saved: frame_skip_2.mp4
Average FPS = 14.72
Min FPS = 10.06
Max FPS = 17.05


In [ ]:
results_list.append({
    "Technique": "Frame Skip 1",
    "Resolution": "416x416",
    "Frame Skip": "No",
    "Average FPS": skip_1_result["avg_fps"],
    "Min FPS": skip_1_result["min_fps"],
    "Max FPS": skip_1_result["max_fps"]
})

results_list.append({
    "Technique": "Frame Skip 2",
    "Resolution": "416x416",
    "Frame Skip": "Every 2nd Frame",
    "Average FPS": skip_2_result["avg_fps"],
    "Min FPS": skip_2_result["min_fps"],
    "Max FPS": skip_2_result["max_fps"]
})

In [ ]:
results_df = pd.DataFrame(results_list)

results_df

,Technique,Resolution,Frame Skip,Average FPS,Min FPS,Max FPS
0,Baseline,640x640,No,6.69,1.69,8.22
1,Resolution 640,640x640,No,6.48,1.12,7.70
2,Resolution 416,416x416,No,14.12,3.52,17.13
3,Resolution 320,320x320,No,21.91,10.92,25.85
4,Frame Skip 1,416x416,No,15.30,1.78,17.26
5,Frame Skip 2,416x416,Every 2nd Frame,14.72,10.06,17.05


In [ ]:
results_df.to_csv("fps_results.csv", index=False)

print("FPS results saved successfully!")

FPS results saved successfully!


In [ ]:
best_result = results_df.loc[
    results_df["Average FPS"].idxmax()
]

print("Best Configuration So Far:")
print(best_result)

Best Configuration So Far:
Technique      Resolution 320
Resolution            320x320
Frame Skip                 No
Average FPS             21.91
Min FPS                 10.92
Max FPS                 25.85
Name: 3, dtype: object


ONNX export + ONNX Runtime inference

In [ ]:
# Export YOLOv8n to ONNX format

model.export(
    format="onnx",
    opset=12
)

print("ONNX export completed!")

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.71'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 10 packages in 570ms
Prepared 2 packages in 93ms
Installed 2 packages in 13ms
 + colorama==0.4.6
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 1.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 12...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 3.0s, saved as 'yolov8n.onnx' (12.3 MB)

Export complete (3.8s)
Results saved to /content/yolov8n.onnx
Predict:         yolo predict task=detect model=

In [ ]:
import onnxruntime as ort

onnx_model_path = "/content/yolov8n.onnx"

session = ort.InferenceSession(
    onnx_model_path,
    providers=["CPUExecutionProvider"]
)

print("ONNX Runtime model loaded!")

ONNX Runtime model loaded!


In [ ]:
def run_onnx_inference(
    model,
    video_path,
    output_path,
    img_size=416,
    frame_skip=1
):

    cap = cv2.VideoCapture(video_path)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_input = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out = cv2.VideoWriter(
        output_path,
        fourcc,
        fps_input,
        (width, height)
    )

    fps_values = []
    frame_count = 0

    start_total = time.time()

    while True:

        success, frame = cap.read()

        if not success:
            break

        frame_count += 1

        if frame_count % frame_skip != 0:
            continue

        start_time = time.time()

        # ONNX inference using ultralytics
        results = model.predict(
            source=frame,
            imgsz=img_size,
            device="cpu",
            verbose=False
        )

        annotated_frame = results[0].plot()

        end_time = time.time()

        fps = 1 / (end_time - start_time)
        fps_values.append(fps)

        cv2.putText(
            annotated_frame,
            f"FPS: {fps:.2f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,255,0),
            2
        )

        out.write(annotated_frame)

    total_time = time.time() - start_total

    cap.release()
    out.release()

    avg_fps = np.mean(fps_values)
    min_fps = np.min(fps_values)
    max_fps = np.max(fps_values)

    print("ONNX processing completed!")

    return {
        "resolution": img_size,
        "frame_skip": frame_skip,
        "avg_fps": round(avg_fps,2),
        "min_fps": round(min_fps,2),
        "max_fps": round(max_fps,2),
        "processing_time_sec": round(total_time,2)
    }

In [ ]:
onnx_model = YOLO("yolov8n.onnx")

print("YOLO ONNX model loaded successfully!")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
YOLO ONNX model loaded successfully!


In [ ]:
onnx_result = run_onnx_inference(
    model=onnx_model,
    video_path=video_path,
    output_path="onnx_output.mp4",
    img_size=640,
    frame_skip=2
)

onnx_result

ONNX processing completed!


{'resolution': 640,
 'frame_skip': 2,
 'avg_fps': np.float64(6.05),
 'min_fps': np.float64(4.01),
 'max_fps': np.float64(6.67),
 'processing_time_sec': 13.05}

In [ ]:
results_list.append({
    "Technique": "ONNX Runtime",
    "Resolution": "416x416",
    "Frame Skip": "Every 2nd Frame",
    "Average FPS": onnx_result["avg_fps"],
    "Min FPS": onnx_result["min_fps"],
    "Max FPS": onnx_result["max_fps"]
})

In [ ]:
results_df = pd.DataFrame(results_list)

results_df

,Technique,Resolution,Frame Skip,Average FPS,Min FPS,Max FPS
0,Baseline,640x640,No,6.69,1.69,8.22
1,Resolution 640,640x640,No,6.48,1.12,7.70
2,Resolution 416,416x416,No,14.12,3.52,17.13
3,Resolution 320,320x320,No,21.91,10.92,25.85
4,Frame Skip 1,416x416,No,15.30,1.78,17.26
5,Frame Skip 2,416x416,Every 2nd Frame,14.72,10.06,17.05
6,ONNX Runtime,416x416,Every 2nd Frame,6.05,4.01,6.67


In [ ]:
results_df.to_csv(
    "final_fps_results.csv",
    index=False
)

print("Final results saved!")

Final results saved!


In [ ]:
best_result = results_df.loc[
    results_df["Average FPS"].idxmax()
]

print("BEST CONFIGURATION")
print(best_result)

BEST CONFIGURATION
Technique      Resolution 320
Resolution            320x320
Frame Skip                 No
Average FPS             21.91
Min FPS                 10.92
Max FPS                 25.85
Name: 3, dtype: object
